# Data Engineer Take-Home Assessment TEG
This assessment contains multiple tasks simulating a real-world data engineering scenario.
You will be working with two datasets:
- **sales.csv** – sales transactions (contains missing values, inconsistent formats, duplicates)
- **products.json** – product catalog with nested structures

**Tasks:**
1. Ingest & store data
2. Clean & transform data
3. Model data for analytics
4. Write a reporting query
5. Document performance/scalability considerations
6. Build a basic SSRS report
---

## Task 1 – Ingestion & Storage
Load the CSV and JSON files 
Handle missing values and ensure correct schema types.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, TimestampType, DecimalType)
from Data.Schemas.sales_schema import sales_read_schema
from Data.Schemas.products_schema import products_read_schema

spark = SparkSession.builder.appName("teg-pipeline").master("local[*]").getOrCreate()

In [47]:
sales = (spark.read
         .option("header", True)
         .schema(sales_read_schema)
         .csv("Data/Raw/sales.csv"))
sales.show(5) 

+-------+----------+----------+--------+-----+
|sale_id|product_id| sale_date|quantity|price|
+-------+----------+----------+--------+-----+
|      1|     101.0|2025-08-01|       2| 10.5|
|      2|     102.0|08/02/2025|       1| 20.0|
|      3|      NULL|2025-08-03|       3| NULL|
|      4|     104.0|2025/08/04|       1| 5.75|
|      4|     104.0|2025/08/04|       1| 5.75|
+-------+----------+----------+--------+-----+
only showing top 5 rows


In [48]:
products = (spark.read
            .option("multiLine", True)
            .schema(products_read_schema)
            .json("Data/Raw/products.json"))
products.show(5, truncate=False) 

+----------+----------+-----------+-----------------------------+
|product_id|name      |category   |details                      |
+----------+----------+-----------+-----------------------------+
|101       |Laptop    |Electronics|{BrandA, 1 year, NULL, NULL} |
|102       |Phone     |Electronics|{BrandB, 2 years, NULL, NULL}|
|104       |Desk Chair|Furniture  |{BrandC, NULL, Leather, NULL}|
|105       |Headphones|Electronics|{BrandD, NULL, NULL, true}   |
+----------+----------+-----------+-----------------------------+



## Task 2 – Data Cleaning & Transformation
- Standardize date formats to `YYYY-MM-DD`
- Remove duplicates
- Fill missing `product_id` with `NULL`
- Ensure prices are numeric with two decimal places

In [ ]:
sales_clean = sales.withColumn("sale_date",
    F.coalesce(
        F.try_to_date("sale_date", "yyyy-MM-dd"),
        F.try_to_date("sale_date", "MM/dd/yyyy"),
        F.try_to_date("sale_date", "yyyy/MM/dd"),
    )
).withColumn("price", 
    F.col("price").cast(DecimalType(10, 2)))

sales_clean.printSchema() 
sales_clean.show()

# sales_clean.filter(F.col("sale_date").isNull()).show()   

root
 |-- sale_id: integer (nullable = true)
 |-- product_id: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: decimal(10,2) (nullable = true)

+-------+----------+----------+--------+-----+
|sale_id|product_id| sale_date|quantity|price|
+-------+----------+----------+--------+-----+
|      1|     101.0|2025-08-01|       2|10.50|
|      2|     102.0|2025-08-02|       1|20.00|
|      3|      NULL|2025-08-03|       3| NULL|
|      4|     104.0|2025-08-04|       1| 5.75|
|      4|     104.0|2025-08-04|       1| 5.75|
|      5|     105.0|2025-08-05|       5| NULL|
+-------+----------+----------+--------+-----+



In [58]:
sales_final = sales_clean.distinct()
sales_final.show()

+-------+----------+----------+--------+-----+
|sale_id|product_id| sale_date|quantity|price|
+-------+----------+----------+--------+-----+
|      4|     104.0|2025-08-04|       1| 5.75|
|      2|     102.0|2025-08-02|       1|20.00|
|      1|     101.0|2025-08-01|       2|10.50|
|      5|     105.0|2025-08-05|       5| NULL|
|      3|      NULL|2025-08-03|       3| NULL|
+-------+----------+----------+--------+-----+



## Task 3 – Data Modeling
Design a star schema for analytics with:
- `fact_sales`
- `dim_product`
- `dim_date`

Provide:
- ER diagram (link or file) - ERD under: Data/ERD/ERD.png
- SQL DDL statements

In [6]:
DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_date;

CREATE TABLE dim_product (
    product_id    NVARCHAR(50)  NOT NULL,
    product_name  NVARCHAR(255) NULL,
    category      NVARCHAR(100) NULL,
    brand         NVARCHAR(100) NULL,
    warranty      NVARCHAR(100) NULL,
    material      NVARCHAR(100) NULL,
    wireless      BIT           NULL,
    CONSTRAINT pk_dim_product PRIMARY KEY (product_id)
);

CREATE TABLE dim_date (
    [date]        DATE          NOT NULL,
    [year]        INT           NOT NULL,
    [month]       INT           NOT NULL,
    [day]         INT           NOT NULL,
    month_name    NVARCHAR(20)  NOT NULL,
    day_name      NVARCHAR(20)  NOT NULL,
    is_weekend    BIT           NOT NULL,
    week_of_year  INT           NOT NULL,
    CONSTRAINT pk_dim_date PRIMARY KEY ([date])
);

CREATE TABLE fact_sales (
    sale_id     NVARCHAR(50)    NOT NULL,
    product_id  NVARCHAR(50)    NULL,
    sale_date   DATE            NOT NULL,
    quantity    INT             NULL,
    price       DECIMAL(10, 2)  NULL,
    revenue     DECIMAL(10, 2)  NULL,
    CONSTRAINT pk_fact_sales PRIMARY KEY (sale_id),
    CONSTRAINT fk_fact_sales_product FOREIGN KEY (product_id) REFERENCES dim_product (product_id),
    CONSTRAINT fk_fact_sales_date    FOREIGN KEY (sale_date)  REFERENCES dim_date ([date])
);

SyntaxError: invalid syntax (1288054016.py, line 1)

## Task 4 – Reporting Query
Write an SQL query to:
> Show total sales amount and total units sold per product category for the last 30 days.

In [ ]:
SELECT COALESCE(p.category, 'Unknown') AS category,
       SUM(f.revenue)  AS total_sales_amount,
       SUM(f.quantity) AS total_units_sold
FROM fact_sales f
LEFT JOIN dim_product p ON f.product_id = p.product_id
WHERE f.sale_date > DATEADD(DAY, -30, (SELECT MAX(sale_date) FROM fact_sales))
GROUP BY COALESCE(p.category, 'Unknown')
ORDER BY total_sales_amount DESC;

## Task 5 – Performance & Scalability
Write a short note explaining how you would:
- Scale the pipeline for 10x data volume
- Make it production-ready (scheduling, monitoring, DQ checks)

In [ ]:
# Your notes here as markdown text

## Task 6 – SSRS Reporting Basics
Using the reporting query from Task 4 as the dataset, outline a basic SSRS report.

Provide:
- A parameterised T-SQL stored procedure for the report dataset (e.g. `@StartDate`, `@EndDate`, `@Category`)
- The SSRS dataset and parameter setup (including a default/cascading parameter)
- A sample RDL snippet or description of the Tablix layout, grouping and totals
- Example SSRS expressions for formatting (e.g. currency, conditional colour on a KPI)
- A short note on how you would deploy the report and schedule a subscription

In [ ]:
# Your SSRS stored procedure / RDL / expressions here